In [1]:
import pandas as pd
import numpy as np
import re
import os
import json
import unicodedata
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Text cleaning
import ftfy                     # fixes mojibake / Unicode glitches
from unidecode import unidecode # ASCII transliteration fallback

# HTTP / URL validation
import requests
from urllib.parse import urlparse

# ML / Embeddings
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer

# Progress
from tqdm import tqdm
tqdm.pandas()

SEED = 42
np.random.seed(SEED)

#   DATASET AUDIT

In [ ]:

df = pd.read_csv("data/raw/products.csv")
df.head()

,title,price,category,gender,img_url,product_url
0,Decathlon Women's Relaxation Yoga Fleece Sweat...,"EGP 1,049.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-rela...
1,Decathlon Women's Fitness Sweatshirt 100 - Pink,EGP 699.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...
2,Decathlon Women's Fitness Hoodie 520 - Pink Qu...,"EGP 1,739.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...
3,Decathlon Women's Cropped Fitness Sweatshirt 5...,"EGP 1,159.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-crop...
4,LC Waikiki Crew Neck Daisy Duck Printed Long S...,EGP 329.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/lc-waikiki-crew-neck-...


In [3]:
df_raw = pd.read_csv("data/raw/products.csv", encoding="utf-8", low_memory=False)
df = df_raw.copy()

In [4]:
print(f"  Rows       : {len(df):,}")
print(f"  Columns    : {df.shape[1]}")
print(f"  Columns    : {df.columns.tolist()}")
print()

  Rows       : 5,955
  Columns    : 6
  Columns    : ['title', 'price', 'category', 'gender', 'img_url', 'product_url']



In [5]:
df = df.reset_index(drop=True)
df.insert(0, 'product_id', df.index)

df.head()

,product_id,title,price,category,gender,img_url,product_url
0,0,Decathlon Women's Relaxation Yoga Fleece Sweat...,"EGP 1,049.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-rela...
1,1,Decathlon Women's Fitness Sweatshirt 100 - Pink,EGP 699.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...
2,2,Decathlon Women's Fitness Hoodie 520 - Pink Qu...,"EGP 1,739.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...
3,3,Decathlon Women's Cropped Fitness Sweatshirt 5...,"EGP 1,159.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-crop...
4,4,LC Waikiki Crew Neck Daisy Duck Printed Long S...,EGP 329.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/lc-waikiki-crew-neck-...


In [6]:
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "  None — all columns fully populated")
print()

  None — all columns fully populated



In [7]:
print("── Duplicate Rows ──────────────────────────────")
print(f"  Exact duplicates : {df.duplicated().sum():,}")
print(f"  Duplicate titles : {df.duplicated('title').sum():,}")
print(f"  Duplicate URLs   : {df.duplicated('img_url').sum():,}")
print()

── Duplicate Rows ──────────────────────────────
  Exact duplicates : 0
  Duplicate titles : 1,247
  Duplicate URLs   : 0



In [8]:
print("── Column Dtypes ───────────────────────────────")
print(df.dtypes)
print()

── Column Dtypes ───────────────────────────────
product_id     int64
title            str
price            str
category         str
gender           str
img_url          str
product_url      str
dtype: object



In [9]:
print("── Category Distribution ───────────────────────")
print(df['category'].value_counts())
print()

── Category Distribution ───────────────────────
category
shoes                                417
jeans                                412
belts                                401
watches                              400
wallets                              400
sunglasses                           383
jewelry                              214
tshirts                              208
bags                                 207
dresses                              206
Coats_Jackets_Vests                  205
hoodies_Sweatshirts                  205
Boys_Fashion_Hoodies_Sweatshirts     201
skirts                               200
Jackets_Coats                        200
Boys_Fashion_Jeans_Pants             200
shorts                               199
half_boots                           199
Girls_Fashion_Hoodies_Sweatshirts    199
Girls_Fashion_Jeans_Pants            199
boots                                192
hoodies_sweatshirts                  164
fashion_accessories                  120

In [10]:
print("── Gender Distribution ──────────")
print(df['gender'].value_counts())
print()

── Gender Distribution ──────────
gender
women    2547
men      1800
kids      799
Men       604
Women     205
Name: count, dtype: int64



In [11]:
print("── Price Sample (raw) ─────────")
print(df['price'].head(10).tolist())

── Price Sample (raw) ─────────
['EGP 1,049.00', 'EGP 699.00', 'EGP 1,739.00', 'EGP 1,159.00', 'EGP 329.00', 'EGP 479.00', 'EGP 265.60 - EGP 266.00', 'EGP 318.74', 'EGP 629.00', 'EGP 318.74']


# Deduplication

In [12]:
# Step 1: Drop exact row duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f"After exact-row dedup   : {len(df):,}  (removed {before - len(df):,})")

# Step 2: Drop rows with the same title + price (same product re-listed)
step2 = len(df)
df.drop_duplicates(subset=['title', 'price'], keep='first', inplace=True)
print(f"After title + price dedup : {len(df):,}  (removed {step2 - len(df):,})")

# Step 3: Drop rows with duplicate image URLs
# Duplicate img_url = same product image = same physical product
step3 = len(df)
df.drop_duplicates(subset=['img_url'], keep='first', inplace=True)
print(f"After img_url dedup     : {len(df):,}  (removed {step3 - len(df):,})")

After exact-row dedup   : 5,955  (removed 0)
After title + price dedup : 5,281  (removed 674)
After img_url dedup     : 5,281  (removed 0)


# Price Cleaning & Normalization

In [13]:
def parse_price(price_str):
    """
    Parses raw price strings into (price_min, price_max, price_egp).
    Returns (NaN, NaN, NaN) if unparseable.
    """
    if pd.isna(price_str) or str(price_str).strip() == '':
        return np.nan, np.nan, np.nan

    # Remove currency symbol and extra whitespace
    cleaned = re.sub(r'[EGP\s]+', ' ', str(price_str)).strip()

    # Extract all numeric values (handles commas inside numbers)
    numbers = re.findall(r'[\d,]+\.?\d*', cleaned)
    numbers = [float(n.replace(',', '')) for n in numbers]

    if len(numbers) == 0:
        return np.nan, np.nan, np.nan
    elif len(numbers) == 1:
        v = numbers[0]
        return v, v, v
    else:
        # Range: use min, max, midpoint
        lo, hi = min(numbers), max(numbers)
        return lo, hi, round((lo + hi) / 2, 2)


# Apply parser
price_parsed = df['price'].apply(parse_price)
df['price_min_egp'] = [p[0] for p in price_parsed]
df['price_max_egp'] = [p[1] for p in price_parsed]
df['price_egp']     = [p[2] for p in price_parsed]   # canonical price

# Flag price ranges (useful feature for recommendation diversity)
df['is_price_range'] = (df['price_min_egp'] != df['price_max_egp']).astype(int)

# Price validity check
invalid_prices = df[(df['price_egp'].isna()) | (df['price_egp'] <= 0)]
print(f"Invalid/zero prices : {len(invalid_prices):,}")

# Normalize price to [0, 1] for ML models
scaler = MinMaxScaler()
valid_mask = df['price_egp'].notna()
df.loc[valid_mask, 'price_normalized'] = scaler.fit_transform(
    df.loc[valid_mask, 'price_egp'].values.reshape(-1, 1)
)

Invalid/zero prices : 0


In [14]:
df['price_egp'].describe()

count     5281.000000
mean       787.030100
std       1118.658412
min         21.000000
25%        279.000000
50%        475.000000
75%        849.000000
max      17260.000000
Name: price_egp, dtype: float64

In [15]:
# Price bucket (for chatbot natural language + recommendation filters)
def price_bucket(p):
    if pd.isna(p):      return 'unknown'
    elif p < 300:       return 'budget'       # < 300 EGP
    elif p < 800:       return 'mid-range'    # 300–799 EGP
    elif p < 2000:      return 'premium'      # 800–1999 EGP
    else:               return 'luxury'       # 2000+ EGP

df['price_bucket'] = df['price_egp'].apply(price_bucket)

print("\nPrice bucket distribution:")
print(df['price_bucket'].value_counts())
print()

print("Price stats (EGP):")
print(df['price_egp'].describe().round(2))

df.head()


Price bucket distribution:
price_bucket
mid-range    2319
budget       1569
premium       960
luxury        433
Name: count, dtype: int64

Price stats (EGP):
count     5281.00
mean       787.03
std       1118.66
min         21.00
25%        279.00
50%        475.00
75%        849.00
max      17260.00
Name: price_egp, dtype: float64


,product_id,title,price,category,gender,img_url,product_url,price_min_egp,price_max_egp,price_egp,is_price_range,price_normalized,price_bucket
0,0,Decathlon Women's Relaxation Yoga Fleece Sweat...,"EGP 1,049.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-rela...,1049.0,1049.0,1049.0,0,0.059632,premium
1,1,Decathlon Women's Fitness Sweatshirt 100 - Pink,EGP 699.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,699.0,699.0,699.0,0,0.039329,mid-range
2,2,Decathlon Women's Fitness Hoodie 520 - Pink Qu...,"EGP 1,739.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,1739.0,1739.0,1739.0,0,0.099658,premium
3,3,Decathlon Women's Cropped Fitness Sweatshirt 5...,"EGP 1,159.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-crop...,1159.0,1159.0,1159.0,0,0.066013,premium
4,4,LC Waikiki Crew Neck Daisy Duck Printed Long S...,EGP 329.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/lc-waikiki-crew-neck-...,329.0,329.0,329.0,0,0.017866,mid-range


# Category Normalization

In [16]:
CATEGORY_MAP = {
    # Tops
    'tshirts':                          ('t-shirts',        'tops'),
    'shirts':                           ('shirts',          'tops'),
    'hoodies_sweatshirts':              ('hoodies & sweatshirts', 'tops'),
    'boys_fashion_hoodies_sweatshirts': ('hoodies & sweatshirts', 'tops'),
    'girls_fashion_hoodies_sweatshirts':('hoodies & sweatshirts', 'tops'),
    'shorts':                           ('shorts',          'bottoms'),

    # Bottoms
    'jeans':                            ('jeans',           'bottoms'),
    'skirts':                           ('skirts',          'bottoms'),
    'boys_fashion_jeans_pants':         ('jeans & pants',   'bottoms'),
    'girls_fashion_jeans_pants':        ('jeans & pants',   'bottoms'),

    # Outerwear
    'coats_jackets_vests':              ('jackets & coats', 'outerwear'),
    'jackets_coats':                    ('jackets & coats', 'outerwear'),

    # Dresses
    'dresses':                          ('dresses',         'dresses'),

    # Footwear
    'shoes':                            ('shoes',           'footwear'),
    'sneakers':                         ('sneakers',        'footwear'),
    'boots':                            ('boots',           'footwear'),
    'half_boots':                       ('ankle boots',     'footwear'),
    'sandals':                          ('sandals',         'footwear'),

    # Accessories
    'bags':                             ('bags',            'accessories'),
    'belts':                            ('belts',           'accessories'),
    'wallets':                          ('wallets',         'accessories'),
    'sunglasses':                       ('sunglasses',      'accessories'),
    'watches':                          ('watches',         'accessories'),
    'jewelry':                          ('jewelry',         'accessories'),
    'fashion_accessories':              ('fashion accessories', 'accessories'),
}

In [17]:

def normalize_category(raw_cat):
    if pd.isna(raw_cat):
        return 'unknown', 'unknown'
    key = str(raw_cat).strip().lower()
    return CATEGORY_MAP.get(key, (key.replace('_', ' '), 'other'))

cat_normalized = df['category'].apply(normalize_category)
df['category_clean']  = [c[0] for c in cat_normalized]
df['category_group']  = [c[1] for c in cat_normalized]

# Label-encode for ML models
le_cat = LabelEncoder()
df['category_id'] = le_cat.fit_transform(df['category_clean'])

le_group = LabelEncoder()
df['category_group_id'] = le_group.fit_transform(df['category_group'])

print("Normalized categories:")
print(df['category_clean'].value_counts())
print()
print("Category groups:")
print(df['category_group'].value_counts())

df.head(1)

Normalized categories:
category_clean
hoodies & sweatshirts    674
watches                  385
shoes                    383
jackets & coats          364
jeans & pants            361
sunglasses               352
jeans                    348
belts                    330
wallets                  323
jewelry                  212
bags                     194
dresses                  193
boots                    190
skirts                   180
shorts                   170
ankle boots              157
t-shirts                 150
fashion accessories      111
sneakers                 106
shirts                    69
sandals                   29
Name: count, dtype: int64

Category groups:
category_group
accessories    1907
bottoms        1059
tops            893
footwear        865
outerwear       364
dresses         193
Name: count, dtype: int64


,product_id,title,price,category,gender,img_url,product_url,price_min_egp,price_max_egp,price_egp,is_price_range,price_normalized,price_bucket,category_clean,category_group,category_id,category_group_id
0,0,Decathlon Women's Relaxation Yoga Fleece Sweat...,"EGP 1,049.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-rela...,1049.0,1049.0,1049.0,0,0.059632,premium,hoodies & sweatshirts,tops,6,5


# Gender Normalization

In [18]:
GENDER_MAP = {
    'women': 'women',
    'woman': 'women',
    'female': 'women',
    'men': 'men',
    'man': 'men',
    'male': 'men',
    'kids': 'kids',
    'boys': 'kids',
    'girls': 'kids',
    'children': 'kids',
    'unisex': 'unisex',
    'all': 'unisex',
}

def normalize_gender(raw_gender):
    if pd.isna(raw_gender):
        return 'unknown'
    key = str(raw_gender).strip().lower()
    return GENDER_MAP.get(key, 'unknown')

df['gender_clean'] = df['gender'].apply(normalize_gender)

# Also infer gender from category name for kids items (cross-validation)
def infer_gender_from_category(row):
    if row['gender_clean'] != 'unknown':
        return row['gender_clean']
    cat = str(row['category']).lower()
    if 'boys' in cat:
        return 'kids'
    elif 'girls' in cat:
        return 'kids'
    elif 'women' in cat:
        return 'women'
    elif 'men' in cat:
        return 'men'
    return 'unknown'

df['gender_clean'] = df.apply(infer_gender_from_category, axis=1)

# Label-encode
le_gender = LabelEncoder()
df['gender_id'] = le_gender.fit_transform(df['gender_clean'])

print("Normalized gender distribution:")
print(df['gender_clean'].value_counts())

Normalized gender distribution:
gender_clean
women    2502
men      2076
kids      703
Name: count, dtype: int64


#  Text Cleaning (Title)

In [19]:
import html

def clean_text(text):
    """
    Full text cleaning pipeline for product titles.
    Returns cleaned string safe for NLP and display.
    """
    if pd.isna(text) or str(text).strip() == '':
        return ''

    # 1. Fix encoding artifacts (mojibake, bad Unicode) cafÃ©" → "café
    text = ftfy.fix_text(str(text))

    # 2. Decode HTML entities (&amp; → &, &nbsp; → space)
    text = html.unescape(text)

    # 3. Strip any residual HTML tags ("<b>Dress</b>" → "Dress")
    text = re.sub(r'<[^>]+>', ' ', text)

    # 4. Normalize Unicode to NFC (composed form) (بيخلي الحروف consistent)
    text = unicodedata.normalize('NFC', text)

    # 5. Replace problematic punctuation / special chars (غريبة characters حذف)
    text = re.sub(r'[\u200b\u200c\u200d\ufeff]', '', text)  # zero-width chars
    text = re.sub(r'[\x00-\x1f\x7f]', ' ', text)            # control chars

    # 6. Normalize dashes and hyphens used as separators (تنظيف الشرطات)
    text = re.sub(r'\s*[–—−]\s*', ' - ', text)

    # 7. Collapse multiple spaces / tabs / newlines (إزالة المسافات الزائدة)
    text = re.sub(r'\s+', ' ', text).strip()

    # 8. Remove trailing/leading punctuation clutter (مثل "!!New Arrival!!" → "New Arrival")
    text = re.sub(r'^[^\w]+|[^\w]+$', '', text).strip()

    return text


In [20]:
def title_for_search(text):
    """Lowercase, no punctuation — optimized for search indexing."""
    if not text:
        return ''
    t = text.lower()
    t = re.sub(r'[^\w\s-]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

#الكود ده بيعمل 3 حاجات:
#1 ينضف العنوان
#2️ يعمل نسخة للبحث
#3️ يقيم جودة العنوان

# Apply
df['title_clean']  = df['title'].progress_apply(clean_text) #for chatbot / عرض العناوين بعد التنظيف
df['title_search'] = df['title_clean'].apply(title_for_search) #for search / عرض العناوين بعد التنظيف + تحسينها للبحث

# Title quality features
df['title_word_count']  = df['title_clean'].apply(lambda t: len(t.split())) #word count
df['title_char_count']  = df['title_clean'].apply(len) #character count

# Flag very short / very long titles as potentially noisy أقل من 3 كلمات → too_short //// أكثر من 25 كلمة → ❌ too_long
df['title_quality_flag'] = df['title_word_count'].apply(
    lambda n: 'too_short' if n < 3 else ('too_long' if n > 25 else 'ok')
)

print("Title quality distribution:")
print(df['title_quality_flag'].value_counts())
print()
print("Sample cleaned titles:")
for _, row in df[['title', 'title_clean']].head(5).iterrows():
    print(f"  Raw : {row['title']}")
    print(f"  Clean: {row['title_clean']}")
    print()

100%|██████████| 5281/5281 [00:00<00:00, 12785.58it/s]


Title quality distribution:
title_quality_flag
ok           5204
too_short      53
too_long       24
Name: count, dtype: int64

Sample cleaned titles:
  Raw : Decathlon Women's Relaxation Yoga Fleece Sweatshirt - Mottled Grey
  Clean: Decathlon Women's Relaxation Yoga Fleece Sweatshirt - Mottled Grey

  Raw : Decathlon Women's Fitness Sweatshirt 100 - Pink
  Clean: Decathlon Women's Fitness Sweatshirt 100 - Pink

  Raw : Decathlon Women's Fitness Hoodie 520 - Pink Quartz
  Clean: Decathlon Women's Fitness Hoodie 520 - Pink Quartz

  Raw : Decathlon Women's Cropped Fitness Sweatshirt 520 - Black
  Clean: Decathlon Women's Cropped Fitness Sweatshirt 520 - Black

  Raw : LC Waikiki Crew Neck Daisy Duck Printed Long Sleeve Women's Sweatshirt
  Clean: LC Waikiki Crew Neck Daisy Duck Printed Long Sleeve Women's Sweatshirt



In [21]:
df.head(5)

,product_id,title,price,category,gender,img_url,product_url,price_min_egp,price_max_egp,price_egp,...,category_group,category_id,category_group_id,gender_clean,gender_id,title_clean,title_search,title_word_count,title_char_count,title_quality_flag
0,0,Decathlon Women's Relaxation Yoga Fleece Sweat...,"EGP 1,049.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-rela...,1049.0,1049.0,1049.0,...,tops,6,5,women,2,Decathlon Women's Relaxation Yoga Fleece Sweat...,decathlon women s relaxation yoga fleece sweat...,9,66,ok
1,1,Decathlon Women's Fitness Sweatshirt 100 - Pink,EGP 699.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,699.0,699.0,699.0,...,tops,6,5,women,2,Decathlon Women's Fitness Sweatshirt 100 - Pink,decathlon women s fitness sweatshirt 100 - pink,7,47,ok
2,2,Decathlon Women's Fitness Hoodie 520 - Pink Qu...,"EGP 1,739.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,1739.0,1739.0,1739.0,...,tops,6,5,women,2,Decathlon Women's Fitness Hoodie 520 - Pink Qu...,decathlon women s fitness hoodie 520 - pink qu...,8,50,ok
3,3,Decathlon Women's Cropped Fitness Sweatshirt 5...,"EGP 1,159.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-crop...,1159.0,1159.0,1159.0,...,tops,6,5,women,2,Decathlon Women's Cropped Fitness Sweatshirt 5...,decathlon women s cropped fitness sweatshirt 5...,8,56,ok
4,4,LC Waikiki Crew Neck Daisy Duck Printed Long S...,EGP 329.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/lc-waikiki-crew-neck-...,329.0,329.0,329.0,...,tops,6,5,women,2,LC Waikiki Crew Neck Daisy Duck Printed Long S...,lc waikiki crew neck daisy duck printed long s...,11,70,ok


# Feature Engineering

In [22]:
COLOR_TERMS = [
    # Basic
    'black', 'white', 'red', 'blue', 'green', 'yellow', 'orange', 'purple',
    'pink', 'brown', 'grey', 'gray', 'beige', 'navy', 'ivory', 'cream',
    'gold', 'silver', 'bronze', 'copper', 'teal', 'turquoise', 'cyan',
    'magenta', 'maroon', 'olive', 'lime', 'coral', 'salmon', 'peach',
    'lavender', 'lilac', 'violet', 'indigo', 'khaki', 'tan', 'camel',
    'ecru', 'off-white', 'offwhite', 'charcoal', 'slate', 'stone',
    'burgundy', 'wine', 'rust', 'mustard', 'sand', 'taupe', 'nude',
    'fuchsia', 'rose', 'blush', 'mint', 'sage', 'forest',
    # Multi-word / compound
    'light blue', 'dark blue', 'royal blue', 'baby blue', 'sky blue',
    'light grey', 'dark grey', 'mottled grey', 'light gray', 'dark gray',
    'hot pink', 'pale pink', 'dusty rose', 'blush pink', 'neon green',
    'dark green', 'light green', 'army green', 'hunter green',
    'off white', 'pure white', 'bright white',
    'light brown', 'dark brown', 'chocolate brown',
    'pale yellow', 'neon yellow', 'bright yellow',
    'dark red', 'bright red', 'cherry red',
    'pink quartz', 'quartz',
]

# Sort by length (longest first) so multi-word matches win
COLOR_TERMS_SORTED = sorted(COLOR_TERMS, key=len, reverse=True)
COLOR_PATTERN = re.compile(
    r'\b(' + '|'.join(re.escape(c) for c in COLOR_TERMS_SORTED) + r')\b',
    flags=re.IGNORECASE
)

In [23]:
# This function extracts all color terms found in the title, returning a list of unique colors in the order they appear.
def extract_colors(title):
    """Return list of colors found in the title."""
    if not title:
        return []
    found = COLOR_PATTERN.findall(title.lower())

    seen = set() 
    result = []
    for c in found:
        if c not in seen:
            seen.add(c)
            result.append(c)
    return result

df['colors']        = df['title_clean'].apply(extract_colors)
df['primary_color'] = df['colors'].apply(lambda c: c[0] if c else 'unknown')
df['color_count']   = df['colors'].apply(len)
df['colors_str']    = df['colors'].apply(lambda c: ', '.join(c))

print("Color extraction coverage:")
has_color = (df['primary_color'] != 'unknown').sum()
print(f"  Products with color  : {has_color:,} ({has_color/len(df)*100:.1f}%)")
print()
print("Top 15 colors:")
print(df['primary_color'].value_counts().head(15))
df.head(5)

Color extraction coverage:
  Products with color  : 2,355 (44.6%)

Top 15 colors:
primary_color
unknown      2926
black         738
white         170
brown         156
beige         135
silver        118
blue          112
gold          112
navy           96
grey           80
quartz         57
pink           50
red            47
green          38
dark blue      31
Name: count, dtype: int64


,product_id,title,price,category,gender,img_url,product_url,price_min_egp,price_max_egp,price_egp,...,gender_id,title_clean,title_search,title_word_count,title_char_count,title_quality_flag,colors,primary_color,color_count,colors_str
0,0,Decathlon Women's Relaxation Yoga Fleece Sweat...,"EGP 1,049.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-rela...,1049.0,1049.0,1049.0,...,2,Decathlon Women's Relaxation Yoga Fleece Sweat...,decathlon women s relaxation yoga fleece sweat...,9,66,ok,[mottled grey],mottled grey,1,mottled grey
1,1,Decathlon Women's Fitness Sweatshirt 100 - Pink,EGP 699.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,699.0,699.0,699.0,...,2,Decathlon Women's Fitness Sweatshirt 100 - Pink,decathlon women s fitness sweatshirt 100 - pink,7,47,ok,[pink],pink,1,pink
2,2,Decathlon Women's Fitness Hoodie 520 - Pink Qu...,"EGP 1,739.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,1739.0,1739.0,1739.0,...,2,Decathlon Women's Fitness Hoodie 520 - Pink Qu...,decathlon women s fitness hoodie 520 - pink qu...,8,50,ok,[pink quartz],pink quartz,1,pink quartz
3,3,Decathlon Women's Cropped Fitness Sweatshirt 5...,"EGP 1,159.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-crop...,1159.0,1159.0,1159.0,...,2,Decathlon Women's Cropped Fitness Sweatshirt 5...,decathlon women s cropped fitness sweatshirt 5...,8,56,ok,[black],black,1,black
4,4,LC Waikiki Crew Neck Daisy Duck Printed Long S...,EGP 329.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/lc-waikiki-crew-neck-...,329.0,329.0,329.0,...,2,LC Waikiki Crew Neck Daisy Duck Printed Long S...,lc waikiki crew neck daisy duck printed long s...,11,70,ok,[],unknown,0,


In [24]:
# 1) PATTERN TERMS
PATTERN_TERMS = [
    'printed', 'pattern', 'striped', 'stripes',
    'floral', 'flower', 'graphic', 'logo',
    'check', 'checked', 'plaid',
    'polka dot', 'dots',
    'camouflage', 'camo',
    'abstract', 'animal print',
    'leopard', 'zebra',
    'embroidery', 'embroidered',
    'lace', 'mesh'
]

# نخلي الطويل الأول (مهم جدًا علشان "polka dot")
PATTERN_TERMS = sorted(PATTERN_TERMS, key=len, reverse=True)

# 2) EXTRACT Raw PATTERN
def extract_pattern(title):
    if pd.isna(title) or not title:
        return 'none'
    
    title = title.lower()
    
    for p in PATTERN_TERMS:
        if p in title:
            return p   # أول pattern
    
    return 'none'

df['pattern'] = df['title_clean'].apply(extract_pattern)


print("Pattern distribution:")
print(df['pattern'].value_counts())

has_pattern = (df['pattern'] != 'none').sum()
print(f"\nProducts with pattern: {has_pattern} ({has_pattern/len(df)*100:.1f}%)")



Pattern distribution:
pattern
none            4612
printed          253
lace             185
pattern           87
floral            37
striped           32
embroidered       22
flower            10
mesh               9
plaid              7
leopard            6
embroidery         5
check              5
camouflage         3
graphic            2
animal print       2
dots               1
stripes            1
logo               1
polka dot          1
Name: count, dtype: int64

Products with pattern: 669 (12.7%)


In [25]:
# 3) NORMALIZATION
PATTERN_MAP = {
    'printed': 'printed',
    'pattern': 'printed',

    'floral': 'floral',
    'flower': 'floral',

    'striped': 'striped',
    'stripes': 'striped',

    'embroidery': 'embroidered',
    'embroidered': 'embroidered',

    'check': 'plaid',
    'checked': 'plaid',
    'plaid': 'plaid',

    'dots': 'polka dot',
    'polka dot': 'polka dot',

    'leopard': 'animal print',
    'zebra': 'animal print',
    'animal print': 'animal print',

    'camouflage': 'camo',
    'camo': 'camo',

    'graphic': 'graphic',
    'logo': 'graphic',

    'lace': 'lace',
    'mesh': 'mesh'
}


def normalize_pattern(p):
    if pd.isna(p) or p == 'none':
        return 'none'   
    p = str(p).strip().lower()
    return PATTERN_MAP.get(p, 'other')

# Normalized clean pattern for better grouping in analysis and ML models
df['pattern_clean'] = df['pattern'].apply(normalize_pattern)

print("Pattern clean distribution:")
print(df['pattern_clean'].value_counts())

# 4) HAS PATTERN FLAG
df['has_pattern'] = (df['pattern_clean'] != 'none').astype(int)


Pattern clean distribution:
pattern_clean
none            4612
printed          340
lace             185
floral            47
striped           33
embroidered       27
plaid             12
mesh               9
animal print       8
graphic            3
camo               3
polka dot          2
Name: count, dtype: int64


In [26]:
# 5) PATTERN GROUPING
PATTERN_GROUP = {
    'printed': 'patterned',
    'floral': 'patterned',
    'striped': 'patterned',
    'plaid': 'patterned',
    'animal print': 'patterned',
    'polka dot': 'patterned',
    'graphic': 'patterned',

    'lace': 'textured',
    'embroidered': 'textured',
    'mesh': 'textured',

    'camo': 'patterned',
    'none': 'plain'
}
# pattern group for recommendation diversity 
df['pattern_group'] = df['pattern_clean'].map(PATTERN_GROUP).fillna('other')

print("\nPattern group distribution:")
print(df['pattern_group'].value_counts())

# 6) CHECK
df.head(5)


Pattern group distribution:
pattern_group
plain        4612
patterned     448
textured      221
Name: count, dtype: int64


,product_id,title,price,category,gender,img_url,product_url,price_min_egp,price_max_egp,price_egp,...,title_char_count,title_quality_flag,colors,primary_color,color_count,colors_str,pattern,pattern_clean,has_pattern,pattern_group
0,0,Decathlon Women's Relaxation Yoga Fleece Sweat...,"EGP 1,049.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-rela...,1049.0,1049.0,1049.0,...,66,ok,[mottled grey],mottled grey,1,mottled grey,none,none,0,plain
1,1,Decathlon Women's Fitness Sweatshirt 100 - Pink,EGP 699.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,699.0,699.0,699.0,...,47,ok,[pink],pink,1,pink,none,none,0,plain
2,2,Decathlon Women's Fitness Hoodie 520 - Pink Qu...,"EGP 1,739.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,1739.0,1739.0,1739.0,...,50,ok,[pink quartz],pink quartz,1,pink quartz,none,none,0,plain
3,3,Decathlon Women's Cropped Fitness Sweatshirt 5...,"EGP 1,159.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-crop...,1159.0,1159.0,1159.0,...,56,ok,[black],black,1,black,none,none,0,plain
4,4,LC Waikiki Crew Neck Daisy Duck Printed Long S...,EGP 329.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/lc-waikiki-crew-neck-...,329.0,329.0,329.0,...,70,ok,[],unknown,0,,printed,printed,1,patterned


# BRAND EXTRACTION

In [27]:
KNOWN_BRANDS = [
    'LC Waikiki', 'Decathlon', 'Defacto', 'BENAA FASHION',
    'Martina Store', 'Caesar', 'Adidas', 'Nike', 'Puma', 'Reebok',
    'New Balance', 'Under Armour', 'Lacoste', 'Tommy Hilfiger',
    'Calvin Klein', 'Ralph Lauren', 'Zara', 'H&M', 'Mango',
    'Forever 21', 'Pull&Bear', 'Bershka', 'Stradivarius',
    'Skechers', 'Clarks', 'Timberland', 'Vans', 'Converse',
    'Dr. Martens', 'Crocs', 'Birkenstock',
]

# Sort longest first
KNOWN_BRANDS_SORTED = sorted(KNOWN_BRANDS, key=len, reverse=True)

# 🔥 مهم: search مش match
BRAND_PATTERN = re.compile(
    r'\b(' + '|'.join(re.escape(b) for b in KNOWN_BRANDS_SORTED) + r')\b',
    flags=re.IGNORECASE
)

def extract_brand(title):
    if not title:
        return 'unknown'

    m = BRAND_PATTERN.search(title)   
    if m:
        return m.group(1)

    return 'unknown'


df['brand'] = df['title_clean'].apply(extract_brand)

print("Top 20 brands extracted:")
print(df['brand'].value_counts().head(50))


df['is_known_brand'] = (df['brand'] != 'unknown').astype(int)
print(df['is_known_brand'].value_counts())
df.head(5)

Top 20 brands extracted:
brand
unknown          3543
LC Waikiki        884
Defacto           557
Caesar            117
ADIDAS             85
Martina Store      30
Skechers           18
BENAA FASHION      16
Nike               16
Reebok              5
Decathlon           4
Forever 21          2
Under Armour        2
Crocs               1
Puma                1
Name: count, dtype: int64
is_known_brand
0    3543
1    1738
Name: count, dtype: int64


,product_id,title,price,category,gender,img_url,product_url,price_min_egp,price_max_egp,price_egp,...,colors,primary_color,color_count,colors_str,pattern,pattern_clean,has_pattern,pattern_group,brand,is_known_brand
0,0,Decathlon Women's Relaxation Yoga Fleece Sweat...,"EGP 1,049.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-rela...,1049.0,1049.0,1049.0,...,[mottled grey],mottled grey,1,mottled grey,none,none,0,plain,Decathlon,1
1,1,Decathlon Women's Fitness Sweatshirt 100 - Pink,EGP 699.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,699.0,699.0,699.0,...,[pink],pink,1,pink,none,none,0,plain,Decathlon,1
2,2,Decathlon Women's Fitness Hoodie 520 - Pink Qu...,"EGP 1,739.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,1739.0,1739.0,1739.0,...,[pink quartz],pink quartz,1,pink quartz,none,none,0,plain,Decathlon,1
3,3,Decathlon Women's Cropped Fitness Sweatshirt 5...,"EGP 1,159.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-crop...,1159.0,1159.0,1159.0,...,[black],black,1,black,none,none,0,plain,Decathlon,1
4,4,LC Waikiki Crew Neck Daisy Duck Printed Long S...,EGP 329.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/lc-waikiki-crew-neck-...,329.0,329.0,329.0,...,[],unknown,0,,printed,printed,1,patterned,LC Waikiki,1


# Colthing Attribute Extraction  

In [28]:

SLEEVE_PATTERNS = {
    'short sleeve': r'short[\s-]?sleeve',
    'long sleeve':  r'long[\s-]?sleeve',
    'sleeveless':   r'sleeveless|no[\s-]?sleeve',
    '3/4 sleeve':   r'3\s*/\s*4[\s-]?sleeve',
    'half sleeve':  r'half[\s-]?sleeve',
}

FIT_PATTERNS = {
    'slim fit':     r'slim[\s-]?fit',
    'relaxed fit':  r'relaxed[\s-]?fit|loose',
    'regular fit':  r'regular[\s-]?fit',
    'oversized':    r'oversize[d]?|over[\s-]?size',
    'fitted':       r'\bfitted\b',
    'cropped':      r'\bcropped?\b',
    'skinny':       r'\bskinny\b',
    'straight':     r'\bstraight\b',
}

MATERIAL_PATTERNS = {
    'cotton':    r'\bcotton\b',
    'polyester': r'\bpolyester\b',
    'denim':     r'\bdenim\b',
    'fleece':    r'\bfleece\b',
    'knit':      r'\bknit\b|\bknitted\b',
    'leather':   r'\bleather\b',
    'linen':     r'\blinen\b',
    'silk':      r'\bsilk\b',
    'velvet':    r'\bvelvet\b',
    'wool':      r'\bwool\b|\bwoolly\b',
    'nylon':     r'\bnylon\b',
    'spandex':   r'\bspandex\b|\belastic\b',
}


In [29]:

def match_first(title, patterns):
    """Return first matching pattern key, or 'unknown'."""
    if not title:
        return 'unknown'
    t = title.lower()
    for label, pat in patterns.items():
        if re.search(pat, t):
            return label
    return 'unknown'

def match_all(title, patterns):
    """Return list of all matching pattern keys."""
    if not title:
        return []
    t = title.lower()
    return [label for label, pat in patterns.items() if re.search(pat, t)]

df['sleeve_type'] = df['title_clean'].apply(lambda t: match_first(t, SLEEVE_PATTERNS))
df['fit_type']    = df['title_clean'].apply(lambda t: match_first(t, FIT_PATTERNS))
df['materials']   = df['title_clean'].apply(lambda t: match_all(t, MATERIAL_PATTERNS))
df['materials_str'] = df['materials'].apply(lambda m: ', '.join(m) if m else 'unknown')

print("Sleeve type distribution:")
print(df['sleeve_type'].value_counts())
print()

print("Fit type distribution:")
print(df['fit_type'].value_counts())
print()

print("Material coverage:")
has_material = (df['materials_str'] != 'unknown').sum()

print(f"  Products with material : {has_material:,} ({has_material/len(df)*100:.1f}%)")

df.head(5)

Sleeve type distribution:
sleeve_type
unknown         5046
long sleeve      151
short sleeve      68
sleeveless        12
half sleeve        4
Name: count, dtype: int64

Fit type distribution:
fit_type
unknown        4805
regular fit     154
slim fit        103
straight         77
oversized        68
relaxed fit      36
skinny           28
cropped           8
fitted            2
Name: count, dtype: int64

Material coverage:
  Products with material : 1,208 (22.9%)


,product_id,title,price,category,gender,img_url,product_url,price_min_egp,price_max_egp,price_egp,...,pattern,pattern_clean,has_pattern,pattern_group,brand,is_known_brand,sleeve_type,fit_type,materials,materials_str
0,0,Decathlon Women's Relaxation Yoga Fleece Sweat...,"EGP 1,049.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-rela...,1049.0,1049.0,1049.0,...,none,none,0,plain,Decathlon,1,unknown,unknown,[fleece],fleece
1,1,Decathlon Women's Fitness Sweatshirt 100 - Pink,EGP 699.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,699.0,699.0,699.0,...,none,none,0,plain,Decathlon,1,unknown,unknown,[],unknown
2,2,Decathlon Women's Fitness Hoodie 520 - Pink Qu...,"EGP 1,739.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,1739.0,1739.0,1739.0,...,none,none,0,plain,Decathlon,1,unknown,unknown,[],unknown
3,3,Decathlon Women's Cropped Fitness Sweatshirt 5...,"EGP 1,159.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-crop...,1159.0,1159.0,1159.0,...,none,none,0,plain,Decathlon,1,unknown,cropped,[],unknown
4,4,LC Waikiki Crew Neck Daisy Duck Printed Long S...,EGP 329.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/lc-waikiki-crew-neck-...,329.0,329.0,329.0,...,printed,printed,1,patterned,LC Waikiki,1,long sleeve,unknown,[],unknown


#  RICH TEXT METADATA BLOB (for chatbot context)

In [30]:
# Combine all structured fields into a single descriptive sentence
# This is what the chatbot retrieves and feeds into its prompt.

def build_metadata_text(row):
    parts = []
    parts.append(row['title_clean'])
# Only include brand if it's known (avoid "Brand: unknown" clutter)
    if row.get('brand', 'unknown') != 'unknown':
        parts.append(f"Brand: {row['brand']}")

    parts.append(f"Category: {row['category_clean']} ({row['category_group']})")
    parts.append(f"Gender: {row['gender_clean']}")
    parts.append(f"Price: EGP {row['price_egp']:.0f} ({row['price_bucket']})")

    if row['primary_color'] != 'unknown':
        parts.append(f"Color: {row['primary_color']}")

    if row['fit_type'] != 'unknown':
        parts.append(f"Fit: {row['fit_type']}")

    if row['sleeve_type'] != 'unknown':
        parts.append(f"Sleeves: {row['sleeve_type']}")

    if row['materials_str'] != 'unknown':
        parts.append(f"Material: {row['materials_str']}")

    return ' | '.join(parts)

df['metadata_text'] = df.apply(build_metadata_text, axis=1)

print("Sample metadata texts:")
for t in df['metadata_text'].head(1).tolist():
    print('>', t)
    print()
df.head(1)

Sample metadata texts:
> Decathlon Women's Relaxation Yoga Fleece Sweatshirt - Mottled Grey | Brand: Decathlon | Category: hoodies & sweatshirts (tops) | Gender: women | Price: EGP 1049 (premium) | Color: mottled grey | Material: fleece



,product_id,title,price,category,gender,img_url,product_url,price_min_egp,price_max_egp,price_egp,...,pattern_clean,has_pattern,pattern_group,brand,is_known_brand,sleeve_type,fit_type,materials,materials_str,metadata_text
0,0,Decathlon Women's Relaxation Yoga Fleece Sweat...,"EGP 1,049.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-rela...,1049.0,1049.0,1049.0,...,none,0,plain,Decathlon,1,unknown,unknown,[fleece],fleece,Decathlon Women's Relaxation Yoga Fleece Sweat...


#  IMAGE URL VALIDATION

In [31]:
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.gif', '.avif'}
CHECK_NETWORK    = False   # Set True to do live HEAD requests (slow)

def validate_url_structure(url):
    """Returns (is_valid: bool, reason: str)."""
    if pd.isna(url) or str(url).strip() == '':
        return False, 'empty'
    try:
        parsed = urlparse(str(url).strip())
        if parsed.scheme not in ('http', 'https'):
            return False, 'bad_scheme'
        if not parsed.netloc:
            return False, 'no_host'
        path = parsed.path.lower().split('?')[0]
        ext  = os.path.splitext(path)[1]
        if ext and ext not in VALID_EXTENSIONS:
            return False, f'bad_ext:{ext}'
        return True, 'ok'
    except Exception as e:
        return False, f'parse_error:{e}'

validation = df['img_url'].apply(validate_url_structure)
df['img_url_valid']  = [v[0] for v in validation]
df['img_url_reason'] = [v[1] for v in validation]

print("Image URL validation summary:")
print(df['img_url_valid'].value_counts())
print()
print("Failure reasons:")
print(df[~df['img_url_valid']]['img_url_reason'].value_counts())
df.head(5)

Image URL validation summary:
img_url_valid
True    5281
Name: count, dtype: int64

Failure reasons:
Series([], Name: count, dtype: int64)


,product_id,title,price,category,gender,img_url,product_url,price_min_egp,price_max_egp,price_egp,...,pattern_group,brand,is_known_brand,sleeve_type,fit_type,materials,materials_str,metadata_text,img_url_valid,img_url_reason
0,0,Decathlon Women's Relaxation Yoga Fleece Sweat...,"EGP 1,049.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-rela...,1049.0,1049.0,1049.0,...,plain,Decathlon,1,unknown,unknown,[fleece],fleece,Decathlon Women's Relaxation Yoga Fleece Sweat...,True,ok
1,1,Decathlon Women's Fitness Sweatshirt 100 - Pink,EGP 699.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,699.0,699.0,699.0,...,plain,Decathlon,1,unknown,unknown,[],unknown,Decathlon Women's Fitness Sweatshirt 100 - Pin...,True,ok
2,2,Decathlon Women's Fitness Hoodie 520 - Pink Qu...,"EGP 1,739.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,1739.0,1739.0,1739.0,...,plain,Decathlon,1,unknown,unknown,[],unknown,Decathlon Women's Fitness Hoodie 520 - Pink Qu...,True,ok
3,3,Decathlon Women's Cropped Fitness Sweatshirt 5...,"EGP 1,159.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-crop...,1159.0,1159.0,1159.0,...,plain,Decathlon,1,unknown,cropped,[],unknown,Decathlon Women's Cropped Fitness Sweatshirt 5...,True,ok
4,4,LC Waikiki Crew Neck Daisy Duck Printed Long S...,EGP 329.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/lc-waikiki-crew-neck-...,329.0,329.0,329.0,...,patterned,LC Waikiki,1,long sleeve,unknown,[],unknown,LC Waikiki Crew Neck Daisy Duck Printed Long S...,True,ok


# Missing Value Strategy

In [32]:
df.head(100)

,product_id,title,price,category,gender,img_url,product_url,price_min_egp,price_max_egp,price_egp,...,pattern_group,brand,is_known_brand,sleeve_type,fit_type,materials,materials_str,metadata_text,img_url_valid,img_url_reason
0,0,Decathlon Women's Relaxation Yoga Fleece Sweat...,"EGP 1,049.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-rela...,1049.00,1049.00,1049.00,...,plain,Decathlon,1,unknown,unknown,[fleece],fleece,Decathlon Women's Relaxation Yoga Fleece Sweat...,True,ok
1,1,Decathlon Women's Fitness Sweatshirt 100 - Pink,EGP 699.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,699.00,699.00,699.00,...,plain,Decathlon,1,unknown,unknown,[],unknown,Decathlon Women's Fitness Sweatshirt 100 - Pin...,True,ok
2,2,Decathlon Women's Fitness Hoodie 520 - Pink Qu...,"EGP 1,739.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,1739.00,1739.00,1739.00,...,plain,Decathlon,1,unknown,unknown,[],unknown,Decathlon Women's Fitness Hoodie 520 - Pink Qu...,True,ok
3,3,Decathlon Women's Cropped Fitness Sweatshirt 5...,"EGP 1,159.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-crop...,1159.00,1159.00,1159.00,...,plain,Decathlon,1,unknown,cropped,[],unknown,Decathlon Women's Cropped Fitness Sweatshirt 5...,True,ok
4,4,LC Waikiki Crew Neck Daisy Duck Printed Long S...,EGP 329.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/lc-waikiki-crew-neck-...,329.00,329.00,329.00,...,patterned,LC Waikiki,1,long sleeve,unknown,[],unknown,LC Waikiki Crew Neck Daisy Duck Printed Long S...,True,ok
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,102,Defacto Woman Turtle Neck Regular Fit Pullover...,EGP 265.60,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/defacto-woman-turtle-...,265.60,265.60,265.60,...,plain,Defacto,1,unknown,regular fit,[],unknown,Defacto Woman Turtle Neck Regular Fit Pullover...,True,ok
104,104,LC Waikiki Crew Neck Mickey Mouse Printed Wome...,EGP 329.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/lc-waikiki-crew-neck-...,329.00,329.00,329.00,...,patterned,LC Waikiki,1,unknown,unknown,[],unknown,LC Waikiki Crew Neck Mickey Mouse Printed Wome...,True,ok
106,106,V A L Y A Women Hoodie Round Neck - Kobi,EGP 299.99,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/v-a-l-y-a-women-hoodi...,299.99,299.99,299.99,...,plain,unknown,0,unknown,unknown,[],unknown,V A L Y A Women Hoodie Round Neck - Kobi | Cat...,True,ok
107,107,V A L Y A Stylish Women Short Hoodie Round Nec...,EGP 249.99,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/v-a-l-y-a-stylish-wom...,249.99,249.99,249.99,...,plain,unknown,0,unknown,unknown,[],unknown,V A L Y A Stylish Women Short Hoodie Round Nec...,True,ok


In [33]:

print(f"  Rows       : {len(df):,}")
print(f"  Columns    : {df.shape[1]}")
print(f"  Columns    : {df.columns.tolist()}")
print()

  Rows       : 5,281
  Columns    : 41
  Columns    : ['product_id', 'title', 'price', 'category', 'gender', 'img_url', 'product_url', 'price_min_egp', 'price_max_egp', 'price_egp', 'is_price_range', 'price_normalized', 'price_bucket', 'category_clean', 'category_group', 'category_id', 'category_group_id', 'gender_clean', 'gender_id', 'title_clean', 'title_search', 'title_word_count', 'title_char_count', 'title_quality_flag', 'colors', 'primary_color', 'color_count', 'colors_str', 'pattern', 'pattern_clean', 'has_pattern', 'pattern_group', 'brand', 'is_known_brand', 'sleeve_type', 'fit_type', 'materials', 'materials_str', 'metadata_text', 'img_url_valid', 'img_url_reason']



In [34]:
df.head()

,product_id,title,price,category,gender,img_url,product_url,price_min_egp,price_max_egp,price_egp,...,pattern_group,brand,is_known_brand,sleeve_type,fit_type,materials,materials_str,metadata_text,img_url_valid,img_url_reason
0,0,Decathlon Women's Relaxation Yoga Fleece Sweat...,"EGP 1,049.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-rela...,1049.0,1049.0,1049.0,...,plain,Decathlon,1,unknown,unknown,[fleece],fleece,Decathlon Women's Relaxation Yoga Fleece Sweat...,True,ok
1,1,Decathlon Women's Fitness Sweatshirt 100 - Pink,EGP 699.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,699.0,699.0,699.0,...,plain,Decathlon,1,unknown,unknown,[],unknown,Decathlon Women's Fitness Sweatshirt 100 - Pin...,True,ok
2,2,Decathlon Women's Fitness Hoodie 520 - Pink Qu...,"EGP 1,739.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,1739.0,1739.0,1739.0,...,plain,Decathlon,1,unknown,unknown,[],unknown,Decathlon Women's Fitness Hoodie 520 - Pink Qu...,True,ok
3,3,Decathlon Women's Cropped Fitness Sweatshirt 5...,"EGP 1,159.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-crop...,1159.0,1159.0,1159.0,...,plain,Decathlon,1,unknown,cropped,[],unknown,Decathlon Women's Cropped Fitness Sweatshirt 5...,True,ok
4,4,LC Waikiki Crew Neck Daisy Duck Printed Long S...,EGP 329.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/lc-waikiki-crew-neck-...,329.0,329.0,329.0,...,patterned,LC Waikiki,1,long sleeve,unknown,[],unknown,LC Waikiki Crew Neck Daisy Duck Printed Long S...,True,ok


# Download Clean CSV

In [35]:
df.to_csv("cleaned_data.csv", index=False)

#df.to_csv("full_dataset.csv", index=False)
#df_final.to_csv("final_dataset.csv", index=False)

In [50]:
# ─────────────────────────────────────────────────────────────
# DEFINE FINAL COLUMN SCHEMA
FINAL_COLUMNS = [
    'product_id',

    'title',
    'price',
    'img_url',
    'product_url',

    'category_clean',
    'category_group',
    'gender_clean',
    'brand',

    'price_egp',
    'price_bucket',

    'primary_color',
    'materials_str',

    'title_search',
    'metadata_text',
]

# Keep only columns that exist in df
FINAL_COLUMNS = [c for c in FINAL_COLUMNS if c in df.columns]
final_data = df[FINAL_COLUMNS].copy()

print("  FINAL DATASET AUDIT :")
print("═" * 60)
print(f"  Total records    : {len(final_data):,}")
print(f"  Total columns    : {final_data.shape[1]}")
print()

print("── Column Schema ───────────────────────────────")
print(final_data.dtypes.to_string())
print()

final_data.head()

  FINAL DATASET AUDIT :
════════════════════════════════════════════════════════════
  Total records    : 5,281
  Total columns    : 15

── Column Schema ───────────────────────────────
product_id          int64
title                 str
price                 str
img_url               str
product_url           str
category_clean        str
category_group        str
gender_clean          str
brand                 str
price_egp         float64
price_bucket          str
primary_color         str
materials_str         str
title_search          str
metadata_text         str



,product_id,title,price,img_url,product_url,category_clean,category_group,gender_clean,brand,price_egp,price_bucket,primary_color,materials_str,title_search,metadata_text
0,0,Decathlon Women's Relaxation Yoga Fleece Sweat...,"EGP 1,049.00",https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-rela...,hoodies & sweatshirts,tops,women,Decathlon,1049.0,premium,mottled grey,fleece,decathlon women s relaxation yoga fleece sweat...,Decathlon Women's Relaxation Yoga Fleece Sweat...
1,1,Decathlon Women's Fitness Sweatshirt 100 - Pink,EGP 699.00,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,hoodies & sweatshirts,tops,women,Decathlon,699.0,mid-range,pink,unknown,decathlon women s fitness sweatshirt 100 - pink,Decathlon Women's Fitness Sweatshirt 100 - Pin...
2,2,Decathlon Women's Fitness Hoodie 520 - Pink Qu...,"EGP 1,739.00",https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,hoodies & sweatshirts,tops,women,Decathlon,1739.0,premium,pink quartz,unknown,decathlon women s fitness hoodie 520 - pink qu...,Decathlon Women's Fitness Hoodie 520 - Pink Qu...
3,3,Decathlon Women's Cropped Fitness Sweatshirt 5...,"EGP 1,159.00",https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-crop...,hoodies & sweatshirts,tops,women,Decathlon,1159.0,premium,black,unknown,decathlon women s cropped fitness sweatshirt 5...,Decathlon Women's Cropped Fitness Sweatshirt 5...
4,4,LC Waikiki Crew Neck Daisy Duck Printed Long S...,EGP 329.00,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/lc-waikiki-crew-neck-...,hoodies & sweatshirts,tops,women,LC Waikiki,329.0,mid-range,unknown,unknown,lc waikiki crew neck daisy duck printed long s...,LC Waikiki Crew Neck Daisy Duck Printed Long S...


In [ ]:
df.to_csv("final_data.csv", index=False)

In [51]:
final_data.to_csv("final_data.csv", index=False, encoding="utf-8-sig")

In [2]:

df = pd.read_csv("cleaned_data.csv")
df.head()

,product_id,title,price,category,gender,img_url,product_url,price_min_egp,price_max_egp,price_egp,...,pattern_group,brand,is_known_brand,sleeve_type,fit_type,materials,materials_str,metadata_text,img_url_valid,img_url_reason
0,0,Decathlon Women's Relaxation Yoga Fleece Sweat...,"EGP 1,049.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-rela...,1049.0,1049.0,1049.0,...,plain,Decathlon,1,unknown,unknown,['fleece'],fleece,Decathlon Women's Relaxation Yoga Fleece Sweat...,True,ok
1,1,Decathlon Women's Fitness Sweatshirt 100 - Pink,EGP 699.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,699.0,699.0,699.0,...,plain,Decathlon,1,unknown,unknown,[],unknown,Decathlon Women's Fitness Sweatshirt 100 - Pin...,True,ok
2,2,Decathlon Women's Fitness Hoodie 520 - Pink Qu...,"EGP 1,739.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...,1739.0,1739.0,1739.0,...,plain,Decathlon,1,unknown,unknown,[],unknown,Decathlon Women's Fitness Hoodie 520 - Pink Qu...,True,ok
3,3,Decathlon Women's Cropped Fitness Sweatshirt 5...,"EGP 1,159.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-crop...,1159.0,1159.0,1159.0,...,plain,Decathlon,1,unknown,cropped,[],unknown,Decathlon Women's Cropped Fitness Sweatshirt 5...,True,ok
4,4,LC Waikiki Crew Neck Daisy Duck Printed Long S...,EGP 329.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/lc-waikiki-crew-neck-...,329.0,329.0,329.0,...,patterned,LC Waikiki,1,long sleeve,unknown,[],unknown,LC Waikiki Crew Neck Daisy Duck Printed Long S...,True,ok


In [11]:
df['metadata_text'].head(100)

0     Decathlon Women's Relaxation Yoga Fleece Sweat...
1     Decathlon Women's Fitness Sweatshirt 100 - Pin...
2     Decathlon Women's Fitness Hoodie 520 - Pink Qu...
3     Decathlon Women's Cropped Fitness Sweatshirt 5...
4     LC Waikiki Crew Neck Daisy Duck Printed Long S...
                            ...                        
95    Defacto Woman Turtle Neck Regular Fit Pullover...
96    LC Waikiki Crew Neck Mickey Mouse Printed Wome...
97    V A L Y A Women Hoodie Round Neck - Kobi | Cat...
98    V A L Y A Stylish Women Short Hoodie Round Nec...
99    Women's hoodies decorated with heart embroider...
Name: metadata_text, Length: 100, dtype: str

In [5]:
df['metadata_text'].isnull().sum()
df['metadata_text'].str.strip().eq('').sum()

np.int64(0)

In [ ]:
df['metadata_text'].str.contains(r'http|www', regex=True).sum()

np.int64(0)

In [7]:
df['metadata_text'].str.contains(r'<.*?>', regex=True).sum()

np.int64(0)

In [8]:
df['metadata_text'].str.contains(r'[^\w\s]', regex=True).sum()

np.int64(5281)

In [9]:
df['metadata_text'].str.len().describe()

count    5281.000000
mean      147.534937
std        28.482530
min        77.000000
25%       128.000000
50%       145.000000
75%       163.000000
max       342.000000
Name: metadata_text, dtype: float64

In [10]:
df['metadata_text'].duplicated().sum()

np.int64(21)

In [12]:
df['metadata_text'].str.findall(r'[^\w\s]').explode().value_counts().head(10)

metadata_text
:    21862
|    21855
(    10632
)    10597
-     6208
&     1708
'     1476
,      614
.      145
/       71
Name: count, dtype: int64

In [13]:
df.columns.tolist()

['product_id',
 'title',
 'price',
 'category',
 'gender',
 'img_url',
 'product_url',
 'price_min_egp',
 'price_max_egp',
 'price_egp',
 'is_price_range',
 'price_normalized',
 'price_bucket',
 'category_clean',
 'category_group',
 'category_id',
 'category_group_id',
 'gender_clean',
 'gender_id',
 'title_clean',
 'title_search',
 'title_word_count',
 'title_char_count',
 'title_quality_flag',
 'colors',
 'primary_color',
 'color_count',
 'colors_str',
 'pattern',
 'pattern_clean',
 'has_pattern',
 'pattern_group',
 'brand',
 'is_known_brand',
 'sleeve_type',
 'fit_type',
 'materials',
 'materials_str',
 'metadata_text',
 'img_url_valid',
 'img_url_reason']